# VGG16-based DCNN for Alzheimer's Disease Classification

This notebook implements feature extraction and classification with the proposed VGG16-based Deep Convolutional Neural Network. The OTSU-GWWHO segmented ROI images (224×224×3) are passed through the five pretrained VGG16 convolutional blocks. The original fully connected head is replaced with Global Average Pooling → Dense 256 (ReLU) → Dropout 0.5 → Dense 128 (ReLU) → Dropout 0.3 → Softmax 4, which classifies MD, MOD, ND and VMD. The model is trained with two-phase transfer learning on each of the 5 cross-validation folds.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `pandas`, `tensorflow`
- Outputs of `OTSU_GWWHO_segmentation.ipynb` and `kfold_cross_validation.ipynb`
- GPU recommended

## Workflow

1. Build the input pipeline for the segmented ROI images
2. Build the VGG16 backbone with the new classification head
3. Phase 1: freeze all VGG16 blocks and train the head (Adam, lr = 1e-3, batch 32, 20 epochs)
4. Phase 2: unfreeze Blocks 4–5 and fine-tune (Adam, lr = 1e-5, batch 32, up to 30 epochs)
5. Use early stopping (patience 7) and keep the model with the lowest validation loss
6. Predict the test set with each fold model and save the predictions

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

SEGMENTED_DIR = "./outputs/04_segmented"
FOLD_DIR = "./outputs/05_folds"
MODEL_DIR = "./outputs/06_models"
RESULTS_DIR = "./outputs/07_results"

CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]
NUM_CLASSES = 4
IMG_SIZE = 224
N_FOLDS = 5

BATCH_SIZE = 32
PHASE1_LR, PHASE1_EPOCHS = 1e-3, 20
PHASE2_LR, PHASE2_EPOCHS = 1e-5, 30
UNFREEZE_BLOCKS = ("block4", "block5")
PATIENCE = 7
RANDOM_SEED = 42

## Input Pipeline

Loads the segmented ROI images, replicates the grayscale channel to 3 channels, resizes to 224×224 and applies the VGG16 input normalisation.

In [ ]:
# ==========================================================
# INPUT PIPELINE
# ==========================================================

def load_roi(path, label=None):
    """Read ROI -> resize 224x224 -> grayscale to 3 channels -> VGG16 normalisation."""
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(tf.image.grayscale_to_rgb(img))
    if label is None:
        return img
    return img, tf.one_hot(label, NUM_CLASSES)


def make_dataset(df, shuffle=False):
    paths = list(df["roi_path"])
    labels = [CLASS_NAMES.index(l) for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 20000), seed=RANDOM_SEED)
    ds = ds.map(load_roi, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## VGG16-DCNN Architecture

VGG16 backbone (13 convolutional layers of 3×3 filters with ReLU, and five 2×2 max-pooling layers; 64 → 512 filters) followed by the task-specific classification head.

In [ ]:
# ==========================================================
# BUILD VGG16-DCNN
# ==========================================================

def build_vgg16_dcnn():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    backbone = VGG16(include_top=False, weights="imagenet", input_tensor=inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)
    x = layers.Dense(256, activation="relu", name="fc_256")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu", name="fc_128")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="softmax_output")(x)
    return models.Model(inputs, outputs, name="OTSU_GWWHO_VGG16_DCNN")


def set_trainable_blocks(model, trainable_blocks=()):
    """() -> all VGG16 blocks frozen; ('block4','block5') -> blocks 1-3 frozen."""
    for layer in model.layers:
        if layer.name.startswith("block"):
            layer.trainable = layer.name.split("_")[0] in trainable_blocks
        else:
            layer.trainable = True


def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss="categorical_crossentropy", metrics=["accuracy"])


build_vgg16_dcnn().summary()

## Two-Phase Transfer Learning

Phase 1 trains only the new head with all five VGG16 blocks frozen. Phase 2 unfreezes Blocks 4 and 5 and fine-tunes with a lower learning rate. Early stopping on the validation loss keeps the best weights over both phases.

In [ ]:
# ==========================================================
# TWO-PHASE TRAINING
# ==========================================================

def callbacks(ckpt_path, best=None):
    ckpt = tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True)
    if best is not None:
        ckpt.best = best
    return [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                             restore_best_weights=True), ckpt]


def train_fold(k, folds_df):
    random.seed(RANDOM_SEED + k); np.random.seed(RANDOM_SEED + k); tf.random.set_seed(RANDOM_SEED + k)
    train_df = folds_df[folds_df.fold != k]
    val_df = folds_df[(folds_df.fold == k) & (folds_df.is_augmented == 0)]
    train_ds, val_ds = make_dataset(train_df, shuffle=True), make_dataset(val_df)

    ckpt_path = os.path.join(MODEL_DIR, f"vgg16_dcnn_fold{k}.keras")
    model = build_vgg16_dcnn()

    # Phase 1: classification head only
    set_trainable_blocks(model, ())
    compile_model(model, PHASE1_LR)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE1_EPOCHS, callbacks=callbacks(ckpt_path))

    # Phase 2: fine-tune VGG16 blocks 4-5
    set_trainable_blocks(model, UNFREEZE_BLOCKS)
    compile_model(model, PHASE2_LR)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE2_EPOCHS,
                   callbacks=callbacks(ckpt_path, best=min(h1.history["val_loss"])))

    history = pd.concat([pd.DataFrame(h1.history).assign(phase=1),
                         pd.DataFrame(h2.history).assign(phase=2)], ignore_index=True)
    history.to_csv(os.path.join(MODEL_DIR, f"history_fold{k}.csv"), index_label="epoch")
    return tf.keras.models.load_model(ckpt_path)

## Train All Folds and Predict the Test Set

Each fold model predicts the held-out 20% test set. The averaged probabilities of the five fold models are also saved.

In [ ]:
# ==========================================================
# TRAIN 5 FOLDS & PREDICT TEST SET
# ==========================================================

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

folds_df = pd.read_csv(os.path.join(FOLD_DIR, "train_folds.csv"))
test_df = pd.read_csv(os.path.join(SEGMENTED_DIR, "test_segmented.csv"))
y_true = np.array([CLASS_NAMES.index(l) for l in test_df["label"]])


def save_predictions(probs, path):
    out = pd.DataFrame({"image_id": test_df["image_id"], "y_true": y_true, "y_pred": probs.argmax(1)})
    for c, name in enumerate(CLASS_NAMES):
        out[f"prob_{name}"] = probs[:, c]
    out.to_csv(path, index=False)


all_probs = []
for k in range(N_FOLDS):
    model = train_fold(k, folds_df)
    probs = model.predict(make_dataset(test_df))
    save_predictions(probs, os.path.join(RESULTS_DIR, f"test_predictions_fold{k}.csv"))
    all_probs.append(probs)

save_predictions(np.mean(all_probs, axis=0), os.path.join(RESULTS_DIR, "test_predictions_ensemble.csv"))